In [13]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
import pandas as pd
from scipy import stats

from transaction_analysis.data.loader import load_all_transactions, load_users

transactions = load_all_transactions()
users = load_users()

columns_transaction = ["amount_usd", "transaction_type", "mcc", "fraud"]
columns_users = [
    "birth_date",
    "gender",
    "per_capita_income_usd",
    "yearly_income_usd",
    "total_debt_usd",
    "credit_score",
    "num_credit_cards",
]

categorical_transaction = ["transaction_type", "mcc", "fraud"]
categorical_users = ["gender"]

In [15]:
transactions_old, transactions_new = (
    transactions[transactions["date"] < "2015-01-01"],
    transactions[transactions["date"] >= "2015-01-01"],
)
users_old, users_new = users[users["id"] < 1000], users[users["id"] >= 1000]
del transactions, users

In [16]:
transactions_new.head()

,transaction_id,date,client_id,card_id,amount_usd,transaction_type,merchant_id,merchant_city,merchant_state,zip,mcc,errors,fraud
6571667,15471192,2015-01-01 00:01:00,316,2038,69.550003,Chip Transaction,79360,Giddings,TX,78942,5411,NaN,<NA>
6571668,15471193,2015-01-01 00:01:00,1585,339,34.820000,Swipe Transaction,69972,Jacksonville,FL,32222,5814,NaN,False
6571669,15471194,2015-01-01 00:03:00,848,3915,64.400002,Chip Transaction,13051,Harwood,MD,20776,5813,NaN,False
6571670,15471195,2015-01-01 00:04:00,1797,300,47.930000,Chip Transaction,54343,San Leandro,CA,94577,4121,NaN,False
6571671,15471196,2015-01-01 00:05:00,1557,2471,25.750000,Online Transaction,9932,ONLINE,NaN,NaN,5311,NaN,False


In [17]:
def columns_drift(
    dataframe_old: pd.DataFrame,
    dataframe_new: pd.DataFrame,
    columns: list[str],
    categorical_columns: list[str] | None = None,
) -> pd.DataFrame:
    categorical_columns = categorical_columns or []
    results = []
    for column in columns:
        ks_shift = stats.ks_2samp(dataframe_old[column].dropna(), dataframe_new[column].dropna())

        chi2_stat, chi2_pvalue = pd.NA, pd.NA
        if column in categorical_columns:
            contingency = (
                pd.DataFrame(
                    {
                        "old": dataframe_old[column].dropna().value_counts(),
                        "new": dataframe_new[column].dropna().value_counts(),
                    }
                )
                .fillna(0)
                .astype(float)
            )
            chi2_shift = stats.chi2_contingency(contingency)
            chi2_stat, chi2_pvalue = chi2_shift.statistic, chi2_shift.pvalue

        results.append(
            {
                "column": column,
                "ks_stat": ks_shift.statistic,
                "ks_pvalue": ks_shift.pvalue,
                "chi2_stat": chi2_stat,
                "chi2_pvalue": chi2_pvalue,
            }
        )

    return pd.DataFrame(results)


columns_drift(transactions_old, transactions_new, columns_transaction, categorical_transaction)

,column,ks_stat,ks_pvalue,chi2_stat,chi2_pvalue
0,amount_usd,0.005323,2.713139e-82,<NA>,<NA>
1,transaction_type,0.718690,0.000000e+00,7948464.656272,0.0
2,mcc,0.007106,2.641449e-146,3768.561136,0.0
3,fraud,0.000471,7.052317e-01,331.112967,0.0


Jest wzorcowy drift w `transaction_type` oraz `fraud`
Brak pewności w `amount_usd` oraz `mcc` - mały rozbieg przy bardzo wysokiej ilości danych

Ponieważ `transaction_type`, `mcc`, oraz `fraud` są kolumnami kategorycznymi, to do mierzenia drifu należy wykorzystać chi_squared

In [18]:
columns_drift(users_old, users_new, columns_users, categorical_users)

/mnt/shared/courses/1/analiza-danych/TransactionDatasetAnalysis/.venv/lib/python3.14/site-packages/scipy/stats/_axis_nan_policy.py:592: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)


,column,ks_stat,ks_pvalue,chi2_stat,chi2_pvalue
0,birth_date,0.017,0.998723,<NA>,<NA>
1,gender,0.002,1.000000,0.002001,0.964325
2,per_capita_income_usd,0.081,0.002818,<NA>,<NA>
3,yearly_income_usd,0.055,0.097103,<NA>,<NA>
4,total_debt_usd,0.065,0.029225,<NA>,<NA>
5,credit_score,0.030,0.759370,<NA>,<NA>
6,num_credit_cards,0.034,0.610166,<NA>,<NA>


Drift jest tylko w `per_capita_income_usd` oraz `total_debt_usd`; `yearly_income_usd` jest na granicy